# M3 — Recomendador por cluster (comportamental)

Este notebook implementa un recomendador que, dado un usuario, devuelve las 5 categorías de producto
más afines a su cluster. Las recomendaciones se calculan a nivel de cluster, no de persona: todos los
miembros de un mismo segmento reciben el mismo top-5.

- Usuario con historial: se usa su segmento comportamental M1 (notebook `06`), que agrupa a usuarios con
  patrones de clic similares.
- Usuario nuevo (cold-start): se usa su cluster demográfico (notebook `05`), que no requiere historial.

El recomendador trabaja a nivel de cluster porque el historial individual es escaso (mediana ≈ 1 evento por
usuario). En la sección 5 se ajusta el peso `α` de la mezcla `α·individual + (1−α)·cluster` sobre validación
y el óptimo resulta α = 0, de modo que el modelo final sirve directamente el top-5 del cluster sin término
individual.

## Salidas

- `data/processed/cluster_recommendations.csv` — top-5 por cluster.
- `data/processed/recommendations.csv` — top-5 por usuario, obtenido por *lookup* `usuario → cluster`
  para mantener la interfaz por usuario de la API.

Evaluado sin fuga temporal (sección 5.bis), el recomendador por cluster no supera al baseline de
popularidad: HR@5 cluster ≈ 0,59 frente a popularidad ≈ 0,63. La ventaja observada previamente
(HR@5 ≈ 0,69) provenía de una fuga temporal, ya que el segmento de `06` se calcula con todo el historial,
incluido el periodo de test. El recomendador conserva valor operativo (recomendaciones interpretables y
homogéneas por segmento, válidas en cold-start), pero no mejora estadísticamente al baseline de
popularidad para predecir el siguiente clic.

## 0 · Librerías y rutas

In [ ]:
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import wilcoxon

# Pipeline de segmentación sin fuga (réplica de 06): log1p -> scaler -> PCA -> KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Anade src/ al path y reutiliza el helper compartido (mismo patron que 01-04)
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(_root / "src"))
from tfm.utils import find_project_root  # noqa: E402

PROCESSED_PATH = find_project_root() / "data" / "processed"
np.random.seed(42)
print("Carpeta de datos:", PROCESSED_PATH)

## 1 · Cargamos eventos, segmento M1 (comportamental) y cluster demográfico

In [ ]:
events = pd.read_csv(PROCESSED_PATH / "events.csv")
events["timestamp"] = pd.to_datetime(events["timestamp"])

# product_new (categoría) y sector — el sector lo usamos para reconstruir el segmento sin fuga (06)
products = pd.read_csv(PROCESSED_PATH / "products.csv")[["id_product", "product_new", "sector"]]
segmentos = pd.read_csv(PROCESSED_PATH / "users_segmented.csv")[["id_user", "segment"]]
clusters = pd.read_csv(PROCESSED_PATH / "users_demo_segments.csv")[["id_user", "demo_cluster"]]

# Solo clicks y opens; target = 1 si hubo click
events = events[events["event_type"].isin(["click", "open"])].copy()
events["target"] = (events["event_type"] == "click").astype(int)

# Juntamos: cada evento con su categoría, su sector, su segmento y su cluster
df = events[["id_user", "id_product", "target", "timestamp"]].copy()
df = df.merge(products, on="id_product", how="left")
df = df.merge(segmentos, on="id_user", how="left")
df = df.merge(clusters, on="id_user", how="left")
df = df.dropna(subset=["product_new"]).reset_index(drop=True)

# Lista de todas las categorías (en orden fijo) y de los segmentos
ALL_CATS = sorted(df["product_new"].unique())
ALL_SEG = sorted(df["segment"].dropna().unique())
print("Eventos:", df.shape, "| categorías:", len(ALL_CATS), "| segmentos:", len(ALL_SEG))

## 2 · Dividimos en train / validación / test (por tiempo)

- **train** (70 % más antiguo): construye las señales.
- **validación** (10 %): elige el mejor α.
- **test** (20 % más reciente): la evaluación final.

In [ ]:
df = df.sort_values("timestamp").reset_index(drop=True)
corte_1 = int(len(df) * 0.70)
corte_2 = int(len(df) * 0.80)

train = df.iloc[:corte_1].copy()
valid = df.iloc[corte_1:corte_2].copy()
test = df.iloc[corte_2:].copy()

print("train:", len(train), "| valid:", len(valid), "| test:", len(test))
print("hasta:", train["timestamp"].max().date(), "/", valid["timestamp"].max().date(), "/", test["timestamp"].max().date())

## 3 · Construimos las señales (solo con datos de train)

- **afinidad_segmento:** tasa de clicks por (segmento, categoría), normalizada por fila.
- **individual:** tasa de clicks de cada usuario por categoría (su historial).
- **popularidad** global y **cluster demográfico** (para baselines y cold-start).

In [ ]:
def construir_senales(tr):
    # --- afinidad por segmento: tasa de clicks de cada segmento en cada categoría ---
    agg = tr.groupby(["segment", "product_new"])["target"].agg(n_eventos="count", n_clicks="sum")
    agg["tasa_click"] = agg["n_clicks"] / agg["n_eventos"]
    matriz = agg.reset_index().pivot(index="segment", columns="product_new", values="tasa_click")
    matriz = matriz.reindex(index=ALL_SEG, columns=ALL_CATS).fillna(0)
    # normalizamos cada fila para que sume 1 (evitando dividir por 0)
    suma_filas = matriz.sum(axis=1).replace(0, 1)
    afinidad_segmento = matriz.div(suma_filas, axis=0)

    # --- individual: tasa de clicks de cada usuario por categoría ---
    individual = (tr.groupby(["id_user", "product_new"])["target"].mean()
                  .reset_index().pivot(index="id_user", columns="product_new", values="target"))

    # --- popularidad global: nº de clicks por categoría, normalizado a [0, 1] ---
    conteo = tr[tr["target"] == 1]["product_new"].value_counts()
    popularidad = np.array([conteo.get(cat, 0) for cat in ALL_CATS], dtype=float)
    popularidad = popularidad / (popularidad.max() + 1e-9)

    # --- afinidad por cluster demográfico ---
    conteo_cluster = (tr[tr["target"] == 1].groupby(["demo_cluster", "product_new"]).size()
                      .unstack(fill_value=0).reindex(columns=ALL_CATS, fill_value=0))
    cluster_aff = conteo_cluster.div(conteo_cluster.max(axis=1).replace(0, 1), axis=0)

    return afinidad_segmento, individual, popularidad, cluster_aff


afinidad_segmento, individual, popularidad, cluster_aff = construir_senales(train)

# Mapas usuario -> segmento y usuario -> cluster (para buscar rápido)
usuario_segmento = df.drop_duplicates("id_user").set_index("id_user")["segment"]
usuario_cluster = df.drop_duplicates("id_user").set_index("id_user")["demo_cluster"]
print("Señales construidas. Afinidad segmento:", afinidad_segmento.shape)

## 4 · Funciones para puntuar y para medir

In [ ]:
def matriz_segmento(user_ids, afinidad_seg, popularidad):
    """Para cada usuario, la fila de afinidad de su segmento (o la popularidad si no tiene segmento)."""
    filas = []
    for uid in user_ids:
        seg = usuario_segmento.get(uid)
        if seg in afinidad_seg.index:
            filas.append(afinidad_seg.loc[seg].values)
        else:
            filas.append(popularidad)
    return np.array(filas)


def matriz_hibrida(user_ids, afinidad_seg, individual, popularidad, alpha):
    """score = alpha * individual + (1 - alpha) * segmento.  Si no hay historial, solo segmento."""
    base = matriz_segmento(user_ids, afinidad_seg, popularidad)
    ind = individual.reindex(index=user_ids, columns=ALL_CATS).values   # NaN donde no hay historial
    hay_historial = ~np.isnan(ind)
    resultado = base.copy()
    resultado[hay_historial] = alpha * ind[hay_historial] + (1 - alpha) * base[hay_historial]
    return resultado


def matriz_cluster(user_ids, cluster_aff, popularidad):
    """Para cada usuario, la afinidad de su cluster demográfico (o popularidad)."""
    filas = []
    for uid in user_ids:
        c = usuario_cluster.get(uid, -1)
        if c in cluster_aff.index:
            filas.append(cluster_aff.loc[c].values)
        else:
            filas.append(popularidad)
    return np.array(filas)


# --- métricas de ranking ---
def dcg(relevancias):
    total = 0.0
    for i, rel in enumerate(relevancias):
        total += rel / np.log2(i + 2)
    return total


def ndcg(orden, clicadas, k):
    relevancias = [1 if cat in clicadas else 0 for cat in orden[:k]]
    ideal = dcg([1] * min(len(clicadas), k))
    return dcg(relevancias) / ideal if ideal > 0 else 0.0


def average_precision(orden, clicadas, k):
    aciertos = 0
    suma = 0.0
    for i, cat in enumerate(orden[:k]):
        if cat in clicadas:
            aciertos += 1
            suma += aciertos / (i + 1)
    return suma / min(len(clicadas), k) if clicadas else 0.0


def categorias_clicadas(split):
    """Para cada usuario, el conjunto de categorías que clicó en ese periodo."""
    solo_clicks = split[split["target"] == 1]
    return solo_clicks.groupby("id_user")["product_new"].apply(set)


def evaluar(matriz_scores, user_ids, verdad):
    """Calcula HR@5, HR@10, NDCG@10 y MAP@10 para una lista de usuarios."""
    hr5, hr10, ndcg_list, map_list = [], [], [], []
    for i, uid in enumerate(user_ids):
        orden_idx = np.argsort(-matriz_scores[i])           # categorías de mayor a menor score
        orden = [ALL_CATS[j] for j in orden_idx]
        clicadas = verdad[uid]
        hr5.append(1 if len(clicadas & set(orden[:5])) > 0 else 0)
        hr10.append(1 if len(clicadas & set(orden[:10])) > 0 else 0)
        ndcg_list.append(ndcg(orden, clicadas, 10))
        map_list.append(average_precision(orden, clicadas, 10))
    return {"HR5": np.array(hr5), "HR10": np.array(hr10),
            "NDCG": np.array(ndcg_list), "MAP": np.array(map_list)}


print("Funciones listas.")

## 5 · Elegimos α en validación (con HR@5)

In [ ]:
verdad_valid = categorias_clicadas(valid)
usuarios_valid = list(verdad_valid.index)

mejor_alpha = None
mejor_hr5 = -1
for alpha in [0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]:
    scores = matriz_hibrida(usuarios_valid, afinidad_segmento, individual, popularidad, alpha)
    hr5 = evaluar(scores, usuarios_valid, verdad_valid)["HR5"].mean()
    print(f"  alpha={alpha}  ->  HR@5 validación = {hr5:.4f}")
    if hr5 > mejor_hr5:
        mejor_hr5 = hr5
        mejor_alpha = alpha

ALPHA = mejor_alpha
print(f"\nMejor alpha: {ALPHA}  (HR@5 validación = {mejor_hr5:.4f})")

## 5.bis · Segmento sin fuga temporal (corrección C1)

El segmento M1 de `06` se calcula con todo el historial del usuario, incluido el periodo que aquí
se usa como test. Esto introduce una fuga temporal: la pertenencia al cluster ya incorpora el
comportamiento futuro que se intenta predecir, lo que infla HR@5.

Para una evaluación sin fuga, el segmento se reconstruye usando solo eventos de train, replicando el
pipeline de `06` (`log1p` → `StandardScaler` → `PCA≥90%` → `KMeans k=12`), ajustado únicamente sobre
train. La afinidad cluster→categoría también se calcula solo con train. En producción (sección 7) se
usa el segmento oficial de `06` con todo el historial conocido, lo cual no constituye fuga: en inferencia
es legítimo usar todo lo observado del usuario.

In [ ]:
# ── Reconstrucción del segmento M1 SIN FUGA TEMPORAL (corrección C1) ──
# El segmento de 06 se calcula con TODO el historial, incluido el periodo de test. Para una
# evaluación sin fuga lo reconstruimos usando SOLO eventos de train, replicando el pipeline de 06
# (log1p en conteos -> StandardScaler -> PCA≥90% var -> KMeans k=12), ajustado solo en train.

SECTORES_06 = ["seguros", "energía", "teleco", "motor", "servicios",
               "hogar", "servicios legales", "salud y belleza", "finanzas"]
LOG_COLS_06 = ["n_eventos", "n_clicks", "n_opens", "n_productos"]
K_SEG = 12  # misma k que el segmento M1 final de 06 (cluster_km)


def _colname(s):
    return s.strip().lower().replace(" ", "_").replace("í", "i").replace("é", "e")


def construir_features_comportamiento(ev):
    """Replica los 3 bloques de features de 06 (actividad + sector + producto) sobre `ev`."""
    ev = ev.copy()
    ev["es_click"] = ev["target"]
    ev["es_open"] = 1 - ev["target"]

    # Bloque 1 — actividad general
    fref = ev["timestamp"].max()
    agg = ev.groupby("id_user").agg(
        n_eventos=("target", "count"), n_clicks=("es_click", "sum"), n_opens=("es_open", "sum"),
        n_productos=("id_product", "nunique"), n_sectores=("sector", "nunique"),
        ultima=("timestamp", "max"),
    ).reset_index()
    agg["recencia_dias"] = (fref - agg["ultima"]).dt.days
    agg = agg.drop(columns="ultima")
    agg["click_rate"] = agg["n_clicks"] / agg["n_eventos"]

    # Bloque 2 — preferencia sectorial (frecuencia + click_rate por sector)
    sec = ev[ev["sector"].isin(SECTORES_06)]
    sec_ev = sec.groupby(["id_user", "sector"]).size().unstack(fill_value=0)
    sec_freq = sec_ev.div(sec_ev.sum(axis=1), axis=0)
    sec_freq.columns = ["sec_" + _colname(c) for c in sec_freq.columns]
    sec_cl = sec[sec["target"] == 1].groupby(["id_user", "sector"]).size().unstack(fill_value=0)
    sec_cr = sec_cl.div(sec_ev.replace(0, np.nan)).fillna(0)
    sec_cr.columns = ["cr_" + _colname(c) for c in sec_cr.columns]
    df_sec = sec_freq.join(sec_cr, how="outer").fillna(0).reset_index()

    # Bloque 3 — preferencia de categoría de producto (top-20 del propio conjunto + 'other')
    top = ev["product_new"].value_counts(normalize=True).head(20).index.tolist()
    ev["prod_bucket"] = ev["product_new"].where(ev["product_new"].isin(top), other="other")
    pc = ev.groupby(["id_user", "prod_bucket"]).size().unstack(fill_value=0)
    pf = pc.div(pc.sum(axis=1), axis=0)
    pf.columns = ["prod_" + c for c in pf.columns]
    df_prod = pf.reset_index()

    return agg.merge(df_sec, on="id_user", how="left").merge(df_prod, on="id_user", how="left").fillna(0)


def segmento_sin_fuga(train_ev, k=K_SEG):
    """Ajusta el pipeline de 06 SOLO con train y devuelve la asignación de segmento por usuario."""
    feat = construir_features_comportamiento(train_ev)
    cols = [c for c in feat.columns if c != "id_user"]
    X = feat[cols].values.astype(float).copy()
    log_idx = [cols.index(c) for c in LOG_COLS_06 if c in cols]
    X[:, log_idx] = np.log1p(X[:, log_idx])
    Xs = StandardScaler().fit_transform(X)
    var = np.cumsum(PCA(random_state=42).fit(Xs).explained_variance_ratio_)
    n_pca = int(np.argmax(var >= 0.90)) + 1
    Xp = PCA(n_components=n_pca, random_state=42).fit_transform(Xs)
    labels = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(Xp)
    return pd.Series(labels, index=feat["id_user"], name="seg_lf"), n_pca


usuario_segmento_lf, n_pca_lf = segmento_sin_fuga(train)

# Afinidad segmento_sin_fuga -> categoría, calculada SOLO con train (sin fuga)
tr_lf = train.copy()
tr_lf["seg_lf"] = tr_lf["id_user"].map(usuario_segmento_lf)
agg_lf = (tr_lf.dropna(subset=["seg_lf"]).groupby(["seg_lf", "product_new"])["target"]
          .agg(n="count", c="sum"))
agg_lf["tasa"] = agg_lf["c"] / agg_lf["n"]
mat_lf = (agg_lf.reset_index().pivot(index="seg_lf", columns="product_new", values="tasa")
          .reindex(columns=ALL_CATS).fillna(0))
afinidad_lf = mat_lf.div(mat_lf.sum(axis=1).replace(0, 1), axis=0)


def matriz_segmento_lf(user_ids):
    """Fila de afinidad del segmento SIN FUGA del usuario (o popularidad si no tiene train)."""
    filas = []
    for uid in user_ids:
        s = usuario_segmento_lf.get(uid)
        filas.append(afinidad_lf.loc[s].values if s in afinidad_lf.index else popularidad)
    return np.array(filas)


print(f"Segmento sin fuga: KMeans k={K_SEG} sobre {n_pca_lf} comp. PCA (solo train).")
print(f"Usuarios con segmento train: {usuario_segmento_lf.notna().sum():,}")

## 6 · Evaluación final en test (con tests estadísticos)

In [ ]:
verdad_test = categorias_clicadas(test)
usuarios_test = list(verdad_test.index)

NOMBRE_M3 = "Recomendador por cluster (M1)"

# Estrategias evaluadas sobre los MISMOS usuarios de test.
# El recomendador final sirve el top-5 del CLUSTER (segmento M1, SIN FUGA); no incluye término
# individual porque la validación (sección 5) dio α=0.
# Añadimos como referencia el mismo modelo con el segmento de 06 (CON fuga) para medir la inflación.
resultados = {}
resultados["Popularidad"] = evaluar(
    matriz_segmento(usuarios_test, afinidad_segmento, popularidad) * 0 + popularidad, usuarios_test, verdad_test)
resultados["Cluster demo (05)"] = evaluar(
    matriz_cluster(usuarios_test, cluster_aff, popularidad), usuarios_test, verdad_test)
resultados[NOMBRE_M3] = evaluar(
    matriz_segmento_lf(usuarios_test), usuarios_test, verdad_test)
resultados["(ref.) segmento 06 con fuga"] = evaluar(
    matriz_segmento(usuarios_test, afinidad_segmento, popularidad), usuarios_test, verdad_test)

print(f"Usuarios evaluados (con click en test): {len(usuarios_test)}\n")
print(f"{'Método':<30}{'HR@5':>8}{'HR@10':>8}{'NDCG@10':>9}{'MAP@10':>8}")
for nombre, m in resultados.items():
    print(f"{nombre:<30}{m['HR5'].mean():>8.4f}{m['HR10'].mean():>8.4f}{m['NDCG'].mean():>9.4f}{m['MAP'].mean():>8.4f}")

inflacion = resultados["(ref.) segmento 06 con fuga"]["HR5"].mean() - resultados[NOMBRE_M3]["HR5"].mean()
print(f"\nInflación de HR@5 por la fuga temporal del segmento: +{inflacion:.4f}")

In [ ]:
# Contraste de HR@5 del recomendador por cluster (sin fuga) frente a los baselines -> test de McNemar
aciertos_m3 = resultados[NOMBRE_M3]["HR5"]
print("McNemar HR@5 (recomendador por cluster vs baseline):")
for nombre in ["Popularidad", "Cluster demo (05)"]:
    aciertos_base = resultados[nombre]["HR5"]
    # usuarios donde el recomendador acierta y el baseline no (y viceversa)
    cluster_gana = int(((aciertos_m3 == 1) & (aciertos_base == 0)).sum())
    baseline_gana = int(((aciertos_m3 == 0) & (aciertos_base == 1)).sum())
    tabla = [[0, baseline_gana], [cluster_gana, 0]]
    p_valor = mcnemar(tabla, exact=False, correction=True).pvalue
    if p_valor < 0.05:
        veredicto = "el recomendador MEJORA" if cluster_gana > baseline_gana else "el BASELINE es mejor"
    else:
        veredicto = "sin diferencia significativa"
    print(f"  vs {nombre:<18} cluster gana en {cluster_gana}, baseline gana en {baseline_gana}  p={p_valor:.3g}  -> {veredicto}")

# Wilcoxon sobre el NDCG por usuario (cluster vs popularidad)
ndcg_m3 = resultados[NOMBRE_M3]["NDCG"]
ndcg_popularidad = resultados["Popularidad"]["NDCG"]
try:
    _, p_wilcoxon = wilcoxon(ndcg_m3, ndcg_popularidad)
    direccion = "cluster > pop" if ndcg_m3.mean() > ndcg_popularidad.mean() else "pop > cluster"
    print(f"\nWilcoxon NDCG@10 (cluster vs popularidad): p={p_wilcoxon:.3g}  ({direccion})")
except ValueError as e:
    print("Wilcoxon no aplicable:", e)

# Intervalo de confianza del HR@5 del recomendador (bootstrap)
rng = np.random.RandomState(42)
medias_bootstrap = []
for _ in range(2000):
    muestra = rng.randint(0, len(aciertos_m3), len(aciertos_m3))
    medias_bootstrap.append(aciertos_m3[muestra].mean())
lo = np.percentile(medias_bootstrap, 2.5)
hi = np.percentile(medias_bootstrap, 97.5)
print(f"HR@5 recomendador por cluster: {aciertos_m3.mean():.4f}   IC95% [{lo:.4f}, {hi:.4f}]")
print(f"HR@5 popularidad (baseline)  : {resultados['Popularidad']['HR5'].mean():.4f}")

In [ ]:
# Gráfico comparativo
metricas = ["HR5", "HR10", "NDCG", "MAP"]
etiquetas = ["HR@5", "HR@10", "NDCG@10", "MAP@10"]
posiciones = np.arange(len(metricas))
ancho = 0.25
centro = (len(resultados) - 1) / 2

fig, ax = plt.subplots(figsize=(10, 4))
for i, (nombre, m) in enumerate(resultados.items()):
    alturas = [m[met].mean() for met in metricas]
    ax.bar(posiciones + (i - centro) * ancho, alturas, ancho, label=nombre)
ax.set_xticks(posiciones)
ax.set_xticklabels(etiquetas)
ax.set_ylabel("Score")
ax.set_title("Recomendador por cluster (M1) vs baselines (test)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6.bis · Comparación estadística (3 métodos)

Se comparan los tres métodos servibles (popularidad, cluster demográfico y recomendador por cluster M1)
con los siguientes contrastes:

1. Media ± desviación entre usuarios, para reflejar la dispersión además del promedio.
2. Friedman (test global no paramétrico): determina si existe alguna diferencia entre los tres métodos.
   Cada usuario es un bloque y los métodos se rankean por su NDCG@10.
3. Post-hoc Wilcoxon pareado en las 3 parejas, con corrección de Holm por comparaciones múltiples
   (controla el error de familia).
4. McNemar del recomendador frente a cada baseline en HR@5 (binario pareado), también con Holm.

In [ ]:
from itertools import combinations

from scipy.stats import friedmanchisquare, rankdata
from statsmodels.stats.multitest import multipletests


def rango_biserial(a, b):
    """Correlacion rango-biserial pareada = tamano de efecto del Wilcoxon, en [-1, 1]
    (~0,1 pequeno, ~0,3 medio, ~0,5 grande)."""
    d = np.asarray(a, float) - np.asarray(b, float)
    d = d[d != 0]
    if d.size == 0:
        return 0.0
    r = rankdata(np.abs(d))
    return (r[d > 0].sum() - r[d < 0].sum()) / r.sum()

# Métodos REALES a comparar (excluimos la referencia "con fuga", que no es un modelo servible)
metodos = ["Popularidad", "Cluster demo (05)", NOMBRE_M3]

# ── Dispersión: media ± sd entre usuarios (no solo la media) ──
print("Media ± sd entre usuarios de test:")
print(f"{'Método':<30}{'HR@5':>18}{'NDCG@10':>18}")
for nombre in metodos:
    h, n = resultados[nombre]["HR5"], resultados[nombre]["NDCG"]
    print(f"{nombre:<30}{h.mean():>10.4f} ± {h.std():.3f}{n.mean():>9.4f} ± {n.std():.3f}")

# ── 1) Friedman: diferencia global entre los 3 métodos (NDCG@10 por usuario) ──
ndcgs = [resultados[m]["NDCG"] for m in metodos]   # alineados: mismos usuarios, mismo orden
chi2, p_fried = friedmanchisquare(*ndcgs)
print(f"\nFriedman (NDCG@10, 3 métodos, {len(usuarios_test)} usuarios): chi2={chi2:.1f}  p={p_fried:.3g}")
W_kendall = chi2 / (len(usuarios_test) * (len(metodos) - 1))   # tamano de efecto de Friedman (0-1)
print(f"  Tamano de efecto global (W de Kendall): {W_kendall:.3f}  (~0,1 pequeno, ~0,3 medio, ~0,5 grande)")

# ── 2) Post-hoc pareado: Wilcoxon en NDCG@10 para las 3 parejas + corrección Holm ──
pares = list(combinations(metodos, 2))
pvals, etiquetas, signos, efectos = [], [], [], []
for a, b in pares:
    _, p = wilcoxon(resultados[a]["NDCG"], resultados[b]["NDCG"])
    pvals.append(p)
    etiquetas.append(f"{a} vs {b}")
    signos.append(f"({a if resultados[a]['NDCG'].mean() > resultados[b]['NDCG'].mean() else b} mayor)")
    efectos.append(rango_biserial(resultados[a]["NDCG"], resultados[b]["NDCG"]))
rech, p_holm, _, _ = multipletests(pvals, alpha=0.05, method="holm")
print("\nPost-hoc Wilcoxon NDCG@10 + Holm:")
for et, sg, p0, ph, rh, ef in zip(etiquetas, signos, pvals, p_holm, rech, efectos):
    print(f"  {et:<46} p={p0:.2e}  p_holm={ph:.2e}  {'SIG' if rh else 'n.s.'}  |r|={abs(ef):.3f} {sg}")

# ── 3) McNemar HR@5 del recomendador vs cada baseline + corrección Holm ──
pmc, etmc, dirmc = [], [], []
for nombre in ["Popularidad", "Cluster demo (05)"]:
    ab, bb = resultados[NOMBRE_M3]["HR5"], resultados[nombre]["HR5"]
    b01, b10 = int(((ab == 0) & (bb == 1)).sum()), int(((ab == 1) & (bb == 0)).sum())
    pmc.append(mcnemar([[0, b01], [b10, 0]], exact=False, correction=True).pvalue)
    etmc.append(f"cluster vs {nombre}")
    dirmc.append("cluster mejor" if b10 > b01 else "baseline mejor")
rech_mc, p_holm_mc, _, _ = multipletests(pmc, alpha=0.05, method="holm")
print("\nMcNemar HR@5 (recomendador vs baseline) + Holm:")
for et, p0, ph, rh, dr in zip(etmc, pmc, p_holm_mc, rech_mc, dirmc):
    print(f"  {et:<40} p={p0:.2e}  p_holm={ph:.2e}  {'SIG' if rh else 'n.s.'} ({dr})")

## 7 · Recomendaciones finales por cluster

El recomendador calcula el top-5 a nivel de cluster (segmento M1) y lo sirve a todos sus miembros.
Las señales se reconstruyen con todos los datos (en inferencia se usa todo el historial conocido del
usuario, lo cual es legítimo) y se guardan dos tablas:

1. `cluster_recommendations.csv` — el top-5 de cada cluster.
2. `recommendations.csv` — el top-5 de cada usuario, obtenido por *lookup* `usuario → su cluster → top-5`.
   Mantiene la interfaz por usuario para la API; el contenido es el de su cluster.

In [ ]:
# Señales con todos los datos (en inferencia se usa todo el historial conocido)
afinidad_f, individual_f, popularidad_f, cluster_f = construir_senales(df)
todos_usuarios = sorted(df["id_user"].unique())

# Top-5 de popularidad (fallback para usuarios sin segmento asignado)
pop_top5 = [ALL_CATS[j] for j in np.argsort(-popularidad_f)[:5]]


def top5_de_scores(scores):
    """Devuelve (categorías, scores) del top-5 a partir de un vector de scores por categoría."""
    mejores = np.argsort(-scores)[:5]
    return [ALL_CATS[j] for j in mejores], [round(float(scores[j]), 6) for j in mejores]


# 1) Tabla POR CLUSTER (segmento M1): el top-5 que se sirve a todos los miembros del segmento
filas_cluster = []
for seg in ALL_SEG:
    scores = afinidad_f.loc[seg].values if seg in afinidad_f.index else popularidad_f
    recs, scs = top5_de_scores(scores)
    fila = {"cluster_id": seg, "fuente": "segmento_m1"}
    for puesto in range(1, 6):
        fila[f"rec_{puesto}"] = recs[puesto - 1]
        fila[f"score_{puesto}"] = scs[puesto - 1]
    filas_cluster.append(fila)

cluster_recs = pd.DataFrame(filas_cluster)
cluster_recs.to_csv(PROCESSED_PATH / "cluster_recommendations.csv", index=False)
print("Guardado: cluster_recommendations.csv", cluster_recs.shape, "(top-5 por cluster)")

# Diccionarios cluster -> top-5 (para el lookup por usuario)
top5_por_cluster = {r["cluster_id"]: [r[f"rec_{i}"] for i in range(1, 6)] for _, r in cluster_recs.iterrows()}
score5_por_cluster = {r["cluster_id"]: [r[f"score_{i}"] for i in range(1, 6)] for _, r in cluster_recs.iterrows()}

# 2) Por USUARIO: lookup usuario -> su cluster -> top-5 del cluster (compatibilidad de interfaz)
filas = []
for uid in todos_usuarios:
    seg = usuario_segmento.get(uid)
    if seg in top5_por_cluster:
        recs, scs = top5_por_cluster[seg], score5_por_cluster[seg]
    else:
        recs, scs = pop_top5, [0.0] * 5   # usuario sin segmento -> popularidad
    fila = {"id_user": uid}
    for puesto in range(1, 6):
        fila[f"rec_{puesto}"] = recs[puesto - 1]
        fila[f"score_{puesto}"] = scs[puesto - 1]
    filas.append(fila)

recomendaciones = pd.DataFrame(filas)
recomendaciones.to_csv(PROCESSED_PATH / "recommendations.csv", index=False)
print("Guardado: recommendations.csv", recomendaciones.shape, "(vía lookup usuario→cluster)")
display(cluster_recs.head())
recomendaciones.head()

## 8 · Cold-start (usuario nuevo)

Un usuario nuevo no tiene segmento (hace falta historial). Le asignamos su **cluster demográfico** y le
damos las categorías más afines de ese cluster.

In [ ]:
def recomendar_nuevo(demo_cluster, k=5):
    if demo_cluster in cluster_f.index:
        scores = cluster_f.loc[demo_cluster].values
    else:
        scores = popularidad_f
    mejores = np.argsort(-scores)[:k]
    return pd.DataFrame({
        "puesto": range(1, k + 1),
        "product_new": [ALL_CATS[j] for j in mejores],
        "score": np.round(scores[mejores], 4),
    })


print("Ejemplo de cold-start (cluster demográfico = 3):")
recomendar_nuevo(3)

## Resumen de decisiones (M3)

| # | Decisión | Por qué |
|---|----------|---------|
| **Por cluster** | el top-5 se calcula a nivel de cluster (segmento M1) y se sirve a todos sus miembros | El historial individual es escaso (mediana ≈1 evento); la señal útil es la del cluster |
| Sin término individual | α=0 en validación → se elimina el componente `individual` de la producción | Mezclar `α·individual + (1−α)·cluster` no mejora; el óptimo en validación es α=0 |
| Cold-start | usuario nuevo → su **cluster demográfico** (05); sin cluster → popularidad | El cold-start no tiene historial, pero sí perfil demográfico |
| Split temporal | train / validación / test por fecha; α elegido en validación | Sin fuga temporal; α no se ajusta sobre el test |
| **Evaluación sin fuga (C1)** | el segmento se reconstruye con SOLO train (5.bis); producción usa el segmento de 06 con todo el historial | Evita que la pertenencia al cluster incorpore el futuro. La fuga inflaba HR@5 en ≈ +0,10 |
| Tests | HR/NDCG/MAP + McNemar + Wilcoxon + bootstrap | Comparación con significación estadística |

**Conclusión:** sin fuga, el recomendador por cluster no supera a popularidad (HR@5 ≈ 0,59 vs 0,63;
McNemar favorece a popularidad). Su valor es operativo (segmentos interpretables, cold-start), no de
mejora predictiva sobre un baseline de popularidad.

**Salidas:**
- `data/processed/cluster_recommendations.csv` (`cluster_id`, `fuente`, `rec_1..5`, `score_1..5`) — recomendación por cluster.
- `data/processed/recommendations.csv` (`id_user`, `rec_1..5`, `score_1..5`) — por usuario vía *lookup* `usuario → cluster`.